In [49]:
import os
import json
import uuid
from pathlib import Path

import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq
from dotenv import load_dotenv

from tqdm import tqdm

In [50]:
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

print(GROQ_API_KEY[:15] + "...")

gsk_Yr6ak8ydnMP...


In [51]:
client = Groq(
    api_key=GROQ_API_KEY
)

print("Groq Connected Successfully")

Groq Connected Successfully


In [52]:
faculty_folder = Path("faculty")

json_files = sorted(faculty_folder.glob("*.json"))
txt_files = sorted(faculty_folder.glob("*.txt"))

print(f"JSON Files : {len(json_files)}")
print(f"TXT Files  : {len(txt_files)}")

JSON Files : 8
TXT Files  : 7


In [53]:
documents = []

for file in json_files:

    with open(file, "r", encoding="utf-8") as f:

        data = json.load(f)

    # Handle both list and dictionary JSON files
    if isinstance(data, list):

        for faculty in data:

            text = ""

            for key, value in faculty.items():

                text += f"{key}: {value}\n"

            documents.append({
                "source": file.name,
                "text": text.strip()
            })

    elif isinstance(data, dict):

        text = ""

        for key, value in data.items():

            text += f"{key}: {value}\n"

        documents.append({
            "source": file.name,
            "text": text.strip()
        })

print("Total Faculty Profiles:", len(documents))

Total Faculty Profiles: 199


In [54]:
print(documents[0]["source"])
print("-" * 60)
print(documents[0]["text"][:1000])

Artificial_Intelligence_faculty_file.json
------------------------------------------------------------
name: Dr. Muhammad Rafi, PhD
designation: Professor and  HOD
email: muhammad.rafi@nu.edu.pk
extension: 222
profile: https://khi.nu.edu.pk/personnel/dr-muhammad-rafi-phd/


In [55]:
texts = []
metadatas = []
ids = []

for i, doc in enumerate(documents):

    texts.append(doc["text"])

    metadatas.append(
        {
            "source": doc["source"]
        }
    )

    ids.append(f"faculty_{i}")

print("Total Embeddings:", len(texts))

Total Embeddings: 199


In [56]:
embedded_documents = []

for i, doc in enumerate(documents):

    embedded_documents.append(
        {
            "id": f"faculty_{i}",
            "source": doc["source"],
            "text": doc["text"]
        }
    )

print("Total Faculty Documents:", len(embedded_documents))

Total Faculty Documents: 199


In [57]:
print(embedded_documents[0]["source"])
print("-" * 50)
print(embedded_documents[0]["text"])

Artificial_Intelligence_faculty_file.json
--------------------------------------------------
name: Dr. Muhammad Rafi, PhD
designation: Professor and  HOD
email: muhammad.rafi@nu.edu.pk
extension: 222
profile: https://khi.nu.edu.pk/personnel/dr-muhammad-rafi-phd/


In [58]:
embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True
)

Batches: 100%|██████████| 7/7 [00:07<00:00,  1.04s/it]


In [59]:
chroma_client = chromadb.PersistentClient(
    path="chroma_db"
)

In [60]:
collection = chroma_client.get_or_create_collection(
    name="faculty_rag"
)

print("Collection Ready")

Collection Ready


In [61]:
# Delete the old collection completely
try:
    chroma_client.delete_collection("faculty_rag")
    print("Old collection deleted.")
except:
    print("No previous collection found.")

# Create a fresh collection
collection = chroma_client.get_or_create_collection(
    name="faculty_rag"
)

# Store all faculty profiles
for doc in tqdm(embedded_documents):

    embedding = embedding_model.encode(doc["text"]).tolist()

    collection.add(
        ids=[doc["id"]],
        embeddings=[embedding],
        documents=[doc["text"]],
        metadatas=[
            {
                "source": doc["source"]
            }
        ]
    )

print("Finished storing embeddings.")
print("Total vectors:", collection.count())

Old collection deleted.


100%|██████████| 199/199 [00:22<00:00,  8.70it/s]

Finished storing embeddings.
Total vectors: 199


In [62]:
print(collection.count())

199


In [63]:
def retrieve(query, top_k=10):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    return results

In [64]:
results = retrieve(
    "all the phd holders?"
)

In [65]:
for i in range(len(results["documents"][0])):

    print("=" * 80)

    print("Source :", results["metadatas"][0][i]["source"])

    print()

    print(results["documents"][0][i][:600])

    print()

Source : Computer_Science_faculty_file.json

name: Dr. Nouman Durrani, PhD
designation: Associate Professor
email: muhammad.nouman@nu.edu.pk
extension: 133
profile: https://khi.nu.edu.pk/personnel/dr-nouman-durrani-phd/

Source : Computer_Science_faculty_file.json

name: Dr. Abdul Aziz , PhD (On Leave)
designation: Associate Professor
email: abdulaziz@nu.edu.pk
extension: 214
profile: https://khi.nu.edu.pk/personnel/dr-abdul-aziz-phd/

Source : Artificial_Intelligence_faculty_file.json

name: Dr. Kamran Ali, PhD
designation: Assistant Professor
email: kamran.ali@nu.edu.pk
extension: 321
profile: https://khi.nu.edu.pk/personnel/dr-kamran-ali/

Source : Computer_Science_faculty_file.json

name: Dr. Nadeem Kafi, PhD
designation: Assistant Professor
email: nadeem.kafi@nu.edu.pk
extension: 131
profile: https://khi.nu.edu.pk/personnel/dr-nadeem-kafi-phd/

Source : Computer_Science_faculty_file.json

name: Dr. Fahad Samad, PhD
designation: Assistant professor , HoS , HoD (CS)
email: fahad.sam

In [66]:
def build_context(results):
    """
    Combine retrieved chunks into one context string.
    """
    context = ""

    for i, doc in enumerate(results["documents"][0]):
        source = results["metadatas"][0][i]["source"]

        context += f"\n\nSource: {source}\n"
        context += doc

    return context

In [67]:
def create_prompt(question, context):

    return f"""
You are a university faculty assistant.

Use ONLY the information present in the retrieved context.

Instructions:

- Read every retrieved faculty profile.
- If the answer exists, answer it directly.
- If multiple faculty satisfy the question, list them.
- If the answer is not present in the retrieved context, say:
"I don't have enough information to answer that."
- Never guess.
- Never use outside knowledge.

Context:
{context}

Question:
{question}

Answer:
"""

In [68]:
def ask_groq(prompt):

    response = client.chat.completions.create(

        model="llama-3.3-70b-versatile",

        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],

        temperature=0.2
    )

    return response.choices[0].message.content

In [70]:
results = retrieve("Who is the HOD of Cyber Security?")

for i in range(len(results["documents"][0])):
    print("=" * 80)
    print(results["metadatas"][0][i]["source"])
    print()
    print(results["documents"][0][i])

Computer_Science_faculty_file.json

name: Dr. Fahad Samad, PhD
designation: Assistant professor , HoS , HoD (CS)
email: fahad.samad@nu.edu.pk
extension: 134
profile: https://khi.nu.edu.pk/personnel/dr-fahad-samad-phd/
Sciences_Humanities_faculty_file.json

name: Dr. Muhammad Shahzad Shaikh
designation: Associate Professor & HoD
email: shahzad.shaikh@nu.edu.pk
extension: 248
profile: https://khi.nu.edu.pk/personnel/dr-muhammad-shahzad-shaikh/
Artificial_Intelligence_faculty_file.json

name: Dr. Muhammad Rafi, PhD
designation: Professor and  HOD
email: muhammad.rafi@nu.edu.pk
extension: 222
profile: https://khi.nu.edu.pk/personnel/dr-muhammad-rafi-phd/
Management_Sciences_faculty_file.json

name: Dr. Sarfaraz Ahmed Bhutto
designation: Assistant Professor & HoD
email: sarfaraz.bhutto@nu.edu.pk
extension: 104
profile: https://khi.nu.edu.pk/personnel/dr-sarfaraz-ahmed-bhutto/
Management_Sciences_faculty_file.json

name: Dr. Muhammad Saad, PhD (ON LEAVE)
designation: Assistant Professor
emai

In [ ]:
print("=" * 70)
print(" Faculty RAG Chatbot")
print("Type 'exit' or 'quit' to end the chat.")
print("=" * 70)

while True:

    question = input("\nYou: ").strip()

    if question.lower() in ["exit", "quit"]:
        print("\nGoodbye!")
        break

    try:
        # Retrieve relevant chunks
        results = retrieve(question)

        # Build context
        context = build_context(results)

        # Create prompt
        prompt = create_prompt(question, context)

        # Get response from Groq
        answer = ask_groq(prompt)

        # Print answer
        print("\nAssistant:\n")
        print(answer)

        # Print sources
        print("\nSources:")
        sources = sorted(set(m["source"] for m in results["metadatas"][0]))
        for source in sources:
            print(f"  • {source}")

    except Exception as e:
        print(f"\nError: {e}")

    print("\n" + "=" * 70)

 Faculty RAG Chatbot
Type 'exit' or 'quit' to end the chat.



You:  hod of cyber security dept?



Assistant:

I don't have enough information to answer that.

Sources:
  • Artificial_Intelligence_faculty_file.json
  • Computer_Science_faculty_file.json
  • Management_Sciences_faculty_file.json
  • Sciences_Humanities_faculty_file.json




You:  hod of cs dept?



Assistant:

Dr. Fahad Samad, PhD (HoD CS) and also Engr. Dr. Fahad Sherwani, PhD is not the HoD but Dr. Fahad Samad, PhD is mentioned as HoD (CS)

Sources:
  • Artificial_Intelligence_faculty_file.json
  • Computer_Science_faculty_file.json
  • Electrical_Engineering_faculty_file.json
  • Management_Sciences_faculty_file.json
  • Sciences_Humanities_faculty_file.json



In [ ]:
print("\nSources Used:\n")

for metadata in results["metadatas"][0]:
    print("-", metadata["source"])


Sources Used:

- Computer_Science_faculty_file.json
- Sciences_Humanities_faculty_file.json
- Management_Sciences_faculty_file.json
- Computer_Science_faculty_file.txt
- cyber_faculty_file.json
- Cyber_Security_faculty_file.json
- Computer_Science_faculty_file.txt
- Electrical_Engineering_faculty_file.json
- Software_Engineering_faculty_file.json
- Artificial_Intelligence_faculty_file.json
